# HARA Component Ablation Study - LARA-Aligned (27092026)

Re-runs the original five-arm component ablation so that it is directly
comparable with the final LARA study. Everything about the data, the component,
the filter and the training budget has been aligned; the arms now read
`Light_Capacity_LARA` instead of `Light_Capacity_HARA`.

## Arms

| Arm | What it isolates |
|---|---|
| `Base_Residual_Attention` | full HARA: global attention + 4 ResBlocks + multi-scale loss |
| `No_Attention_HARA` | removes the attention block |
| `Plain_Conv_HARA` | removes the residual blocks (attention kept) |
| `Light_Capacity_LARA` | LARA (`variant="pyramid_funnel32"`) - the final proposed model |
| `Heavy_Capacity_HARA` | doubles stem and base width |

## What changed against Final_Ablation_20092026

| Setting | 20092026 | 27092026 |
|---|---|---|
| Data root | `D:\HARA_Final\merged.csv` / `.hdf5` | `D:\STEAD\merged.csv` / `.hdf5` |
| Magnitude filter | `> 3.0` | `> 2.5` (pool 25,091) |
| Component | 0 | **2** (a different physical component) |
| `N_EVAL` | 20 | **200** |
| SNR | pooled over the flattened test set | per-sample, then averaged |
| Light arm | `Light_Capacity_HARA` | `Light_Capacity_LARA` |
| Checkpoints | working directory | `<run>/ckpts/` |
| CR check | none | 50-combination pre-flight + post-run report |

The training recipe, the compression ratios, the architecture of the four HARA
arms and the figure code are unchanged.

## Scope

This study compares **LARA as a whole** against the HARA arms. LARA differs from
them along four axes at once - filled funnel, B=32 stem, six-level pyramid
decoder, and a two-layer refinement head - so it does **not** isolate which
individual component produces the gain. A component-level attribution would
need LARA ablations (funnel off, linear decoder, single head) as extra arms.


In [ ]:
# =============================
# Imports, seeds, device, run mode
# =============================
import os
import sys
import math
import time
import random
import warnings

import h5py
import numpy as np
import pandas as pd
import matplotlib
if os.environ.get("FINAL_ABLATION_HEADLESS") == "1":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ssim
from scipy import stats

if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except (AttributeError, OSError):
        pass
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================
# CONFIG - aligned with Final_LARA_27092026
# =============================
# SMOKE shrinks the whole study to 100 traces / 1 CR / 2 epochs so every arm can be
# exercised end-to-end in a couple of minutes before the real run is launched.
SMOKE = os.environ.get("FINAL_ABLATION_SMOKE", "0") == "1"

# ---- data (identical filter and component to Final_LARA_27092026) ----
CSV_PATH        = r"D:\STEAD\merged.csv"
H5_PATH         = r"D:\STEAD\merged.hdf5"
N_TRACES        = None          # None = all traces passing the filter (~25,091)
MIN_MAGNITUDE   = 2.5           # was 3.0 in Final_Ablation_20092026
MAX_DISTANCE_KM = 60
COMPONENT       = 2             # was 0 -- a DIFFERENT physical component
N_EVAL          = 200           # was 20

# ---- study grid ----
CRS = [2, 3, 5, 10, 15, 20, 30, 50, 60, 100]

BATCH   = 16
WORKERS = 2
SEED    = 42

# ---- training recipe (identical to the LARA 'heavy' recipe) ----
MAX_EPOCHS     = 120
LR             = 0.0002
WEIGHT_DECAY   = 1e-5
GRAD_CLIP      = 1.0
PATIENCE       = 8
SCHED_PATIENCE = 5
MIN_LR         = 1e-6

# ---- verification ----
CR_TOL = 0.001   # relative tolerance on |actual - requested| / requested
ENFORCE_CR_EXACT = True

# ---- output ----
# Anchored to the notebook's own folder, not the working directory, so the run
# writes next to this file wherever it is copied (the GPU machine may have no
# D: drive, and a bare relative path would silently land somewhere else).
def _notebook_dir():
    for var in ("__vsc_ipynb_file__", "__file__"):
        p = globals().get(var)
        if p:
            return os.path.dirname(os.path.abspath(p))
    return os.getcwd()


NOTEBOOK_DIR = _notebook_dir()
RESULTS_DIR  = os.path.join(NOTEBOOK_DIR, "seismic 19 9 2026_LARA_aligned")
CKPT_DIR     = os.path.join(RESULTS_DIR, "ckpts")
# the plotting helpers write figures relative to `results_dir`
results_dir = os.path.join(RESULTS_DIR, "results")
RESULTS_CSV = os.path.join(results_dir, "ablation_study_results.csv")

if SMOKE:
    N_TRACES, CRS, N_EVAL = 100, [10], 10
    RESULTS_DIR = os.path.join(NOTEBOOK_DIR, "seismic 19 9 2026_LARA_aligned_smoke")
    CKPT_DIR    = os.path.join(RESULTS_DIR, "ckpts")
    results_dir = os.path.join(RESULTS_DIR, "results")
    RESULTS_CSV = os.path.join(results_dir, "ablation_study_results.csv")
    MAX_EPOCHS  = 2

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)


def best_file(model_name, cr):
    return os.path.join(CKPT_DIR, f"best_{model_name}_cr{cr}.pth")


print("=" * 68)
print("ABLATION RUN CONFIG (LARA-aligned)")
print("-" * 68)
print(f"  device              : {device}")
print(f"  filter              : earthquake_local, magnitude > {MIN_MAGNITUDE}, "
      f"distance_km <= {MAX_DISTANCE_KM}")
print(f"  component           : {COMPONENT}   (Final_Ablation_20092026 used 0)")
print(f"  eval traces (N_EVAL): {N_EVAL}")
print(f"  CRS                 : {CRS}")
print(f"  recipe              : Adam lr={LR} wd={WEIGHT_DECAY} clip={GRAD_CLIP}, "
      f"max_epochs={MAX_EPOCHS} patience={PATIENCE} sched_patience={SCHED_PATIENCE}")
print(f"  CR tolerance        : {CR_TOL}  (enforce={ENFORCE_CR_EXACT})")
print(f"  notebook dir        : {NOTEBOOK_DIR}   (output is anchored here)")
print(f"  results dir         : {os.path.abspath(RESULTS_DIR)}")
print(f"  checkpoint dir      : {os.path.abspath(CKPT_DIR)}")
print(f"  SMOKE               : {SMOKE}")
print("=" * 68)


## Data

Aligned with `Final_LARA_27092026`: the STEAD merge, `magnitude > 2.5`,
`distance_km <= 60`, **component 2**, and the same 80/10/10 split with
`SEED = 42`. Roughly 25,091 traces qualify.

Component 0 and component 2 are different physical channels on this build
(measured mean correlation -0.07, 0 of 60 traces above 0.90), which is why the
component had to match the LARA study rather than the original ablation.


In [ ]:
# =============================
# Load and filter metadata, then waveforms
# =============================
# NOTE: the component matters.  Final_Ablation_20092026 read column 0; this run
# reads COMPONENT = 2 to match Final_LARA_27092026.  On this STEAD build the two
# channels are different physical components (measured mean correlation -0.07,
# with 0 of 60 traces above 0.90), so mixing them is not defensible.
df = pd.read_csv(CSV_PATH, low_memory=False,
                 usecols=["trace_name", "trace_category", "source_distance_km", "source_magnitude"])
df = df[(df.trace_category == 'earthquake_local') &
        (df.source_magnitude > MIN_MAGNITUDE) &
        (df.source_distance_km <= MAX_DISTANCE_KM)]
trace_names = df['trace_name'].to_list()
if N_TRACES is not None:
    trace_names = trace_names[:N_TRACES]

print(f"[data] filter: earthquake_local, magnitude > {MIN_MAGNITUDE}, "
      f"distance_km <= {MAX_DISTANCE_KM} -> pool = {len(df)}")
print(f"[data] loading {len(trace_names)} traces (component {COMPONENT})")


def load_filtered_stead(hdf5_file, names, group='data', min_len=1500, component=COMPONENT):
    """Load waveforms without normalisation; the Dataset normalises per trace."""
    data = []
    with h5py.File(hdf5_file, 'r') as f:
        for i, name in enumerate(names):
            try:
                waveform = f[group][name][:]
                if waveform.shape[0] >= min_len:
                    data.append(waveform[:min_len, component])
            except KeyError:
                continue
            if (i + 1) % 2000 == 0:
                print(f"  loaded {i + 1}/{len(names)}")
    print(f"[data] loaded waveforms: {len(data)}")
    return np.array(data)


waveforms = load_filtered_stead(H5_PATH, trace_names)
print(f"waveforms shape: {waveforms.shape}")


In [ ]:
# =============================
# 🔹 Memory-Efficient Dataset
# =============================
class EfficientSeismicDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        waveform = self.data[idx]
        z_min = np.min(waveform)
        z_max = np.max(waveform)

        if z_max - z_min > 0:
            waveform = (waveform - z_min) / (z_max - z_min)
        else:
            waveform = np.zeros_like(waveform)

        return torch.tensor(waveform, dtype=torch.float32).unsqueeze(0)

## Shared Building Blocks (from Final_HARA_20092026.py)

In [ ]:
# =============================
# 🔹 SHARED BUILDING BLOCKS
# =============================

class AttentionBlock(nn.Module):
    """Simplified channel attention mechanism - EXACT ABLATION VERSION"""
    def __init__(self, channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, max(4, channels//8), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(max(4, channels//8), channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights


class EnhancedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.elu(out)
        out = self.attention(out)
        return out


class PlainConvBlock(nn.Module):
    """Plain convolutional block without residual connections (for ablation)"""
    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.elu(out)
        out = self.attention(out)
        return out

## HARA backbone and the four HARA arms

Verbatim from the original study so the comparison is not confounded by an
accidental edit. `Light_Capacity_HARA` is still defined - the notebook stays
self-contained and reproducible - but it is **not** in `ABLATION_MODELS`
(see the next cell): `Light_Capacity_LARA` replaces it.


In [ ]:
# =============================
# 🔹 HARA (Heavy Advanced Residual Autoencoder) - Ablation Base
# =============================
class HARA_AblationBase(nn.Module):
    """
    HARA with a TRUE information bottleneck (transferred from Final_HARA_20092026.py).

    Encoder backbone: strided conv -> attention + residual blocks (unchanged).
    Bottleneck: channel reduction -> pool -> flatten -> linear projection
    to a flat 1D latent with EXACTLY 1500//cr values.
    Decoder: linear projection -> upsample + residual blocks -> final conv.

    Ablation knobs (override in subclasses):
      USE_ATTENTION   - False removes every channel-attention block
      USE_RESIDUAL    - False replaces residual blocks with plain conv blocks
      BASE_CHANNELS   - capacity scaling (Light=32, Heavy=128)
      BOTTLE_CHANNELS - bottleneck width (Light=16, Heavy=64)
    """

    BOTTLE_CHANNELS = 32
    BASE_CHANNELS = 64
    USE_ATTENTION = True
    USE_RESIDUAL = True

    def __init__(self, cr, input_len=1500):
        super().__init__()
        self.cr = cr
        self.input_len = input_len
        self.latent_dim = input_len // cr

        self.L = 188 if cr <= 30 else 94
        self.T = min(self.latent_dim, self.L)

        block = EnhancedResidualBlock if self.USE_RESIDUAL else PlainConvBlock
        use_att = self.USE_ATTENTION

        c0 = self.BASE_CHANNELS
        c1 = c0 * 2
        c2 = c0 * 4

        # ---- encoder backbone ----
        encoder_layers = [
            nn.Conv1d(1, c0, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(c0),
            nn.ELU(inplace=True),
        ]
        if use_att:
            encoder_layers.append(AttentionBlock(c0))

        encoder_layers.append(block(c0, c1, stride=2, use_attention=use_att))
        encoder_layers.append(block(c1, c2, stride=2, use_attention=use_att))

        if cr > 30:
            encoder_layers.append(block(c2, c2 * 2, stride=2, use_attention=use_att))
            final_channels = c2 * 2
        else:
            final_channels = c2

        # ---- true compression bottleneck ----
        bottleneck = [
            nn.Conv1d(final_channels, self.BOTTLE_CHANNELS, kernel_size=1),
            nn.ELU(inplace=True),
            nn.AdaptiveAvgPool1d(self.T),
            nn.Flatten(),
            nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_dim),
        ]

        self.encoder = nn.Sequential(*(encoder_layers + bottleneck))

        # ---- decoder ----
        self.upsample_factors = self.calculate_upsample_factors()

        decoder_layers = [
            nn.Linear(self.latent_dim, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)),
            nn.ELU(inplace=True),
        ]

        for factor in self.upsample_factors:
            decoder_layers.extend([
                nn.Upsample(scale_factor=factor, mode='linear', align_corners=False),
                block(self.BOTTLE_CHANNELS, self.BOTTLE_CHANNELS, stride=1, use_attention=use_att),
            ])

        decoder_layers.extend([
            nn.Conv1d(self.BOTTLE_CHANNELS, 1, kernel_size=15, padding=7),
            nn.Sigmoid(),
        ])

        self.decoder = nn.Sequential(*decoder_layers)

        self.test_dimensions()

    # ------------------------------------------------------------------ #
    # helpers
    # ------------------------------------------------------------------ #

    def calculate_upsample_factors(self):
        current_length = self.T
        factors = []
        while current_length < self.input_len:
            remaining_ratio = self.input_len / current_length
            factor = min(2.0, remaining_ratio)
            factors.append(factor)
            current_length = int(current_length * factor)
            if len(factors) >= 5:
                break
        if current_length != self.input_len:
            factors.append(self.input_len / current_length)
        return factors

    def test_dimensions(self):
        try:
            with torch.no_grad():
                test_input = torch.randn(1, 1, self.input_len)
                encoded   = self.encoder(test_input)
                latent_n  = encoded.numel() // encoded.shape[0]

                assert latent_n == self.latent_dim, f"Latent {latent_n} != {self.latent_dim}"

                decoded = self.decoder(encoded)
                if decoded.shape[2] != self.input_len:
                    self.decoder = nn.Sequential(
                        self.decoder,
                        nn.Upsample(size=self.input_len, mode='linear', align_corners=False),
                    )
                    decoded = self.decoder(encoded)

                assert decoded.shape[2] == self.input_len, f"Output {decoded.shape[2]} != {self.input_len}"
                actual_cr = self.input_len / latent_n
                print(f"[OK] {self.__class__.__name__} CR={self.cr}: Input {self.input_len} -> Latent {latent_n} -> Output {decoded.shape[2]} (True CR = {actual_cr:.2f})")
        except Exception as e:
            print(f"[FAIL] {self.__class__.__name__} CR={self.cr}: {e}")
            self.create_fallback_architecture()

    def create_fallback_architecture(self):
        print("Creating fallback architecture...")
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=15, stride=2, padding=7), nn.ELU(),
            nn.Conv1d(64, 128, kernel_size=9, stride=2, padding=4), nn.ELU(),
            nn.Conv1d(128, self.BOTTLE_CHANNELS, kernel_size=1), nn.ELU(),
            nn.AdaptiveAvgPool1d(self.T), nn.Flatten(),
            nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)), nn.ELU(),
            nn.Upsample(size=self.input_len, mode='linear', align_corners=False),
            nn.Conv1d(self.BOTTLE_CHANNELS, 1, kernel_size=15, padding=7), nn.Sigmoid(),
        )
        print("[OK] Fallback architecture created")

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


# =============================
# 🔹 ABLATION 1: BASE MODEL (Full HARA)
# =============================
class Base_Residual_Attention(HARA_AblationBase):
    """Full HARA structure (residual + attention + full capacity)."""
    BOTTLE_CHANNELS = 32
    BASE_CHANNELS = 64
    USE_ATTENTION = True
    USE_RESIDUAL = True


# =============================
# 🔹 ABLATION 2: NO ATTENTION
# =============================
class No_Attention_HARA(HARA_AblationBase):
    """HARA with every channel-attention block removed."""
    USE_ATTENTION = False
    USE_RESIDUAL = True


# =============================
# 🔹 ABLATION 3: PLAIN CONVOLUTION (No Residual)
# =============================
class Plain_Conv_HARA(HARA_AblationBase):
    """HARA with residual connections removed (attention kept)."""
    USE_ATTENTION = True
    USE_RESIDUAL = False


# =============================
# 🔹 ABLATION 4: LIGHT CAPACITY (50% channels)
# =============================
class Light_Capacity_HARA(HARA_AblationBase):
    """HARA with halved channel dimensions."""
    BOTTLE_CHANNELS = 16
    BASE_CHANNELS = 32
    USE_ATTENTION = True
    USE_RESIDUAL = True


# =============================
# 🔹 ABLATION 5: HEAVY CAPACITY (2x channels)
# =============================
class Heavy_Capacity_HARA(HARA_AblationBase):
    """HARA with doubled channel dimensions."""
    BOTTLE_CHANNELS = 64
    BASE_CHANNELS = 128
    USE_ATTENTION = True
    USE_RESIDUAL = True


# =============================
# 🔹 EXACT HARA ALIAS (matches Final_HARA_20092026.py)
# =============================
class HeavyAdvancedResidualAutoencoder(HARA_AblationBase):
    """Alias of the full HARA exactly as in Final_HARA_20092026.py."""
    BOTTLE_CHANNELS = 32
    BASE_CHANNELS = 64
    USE_ATTENTION = True
    USE_RESIDUAL = True

## LARA - the light-capacity arm

`class LARA` below is copied **verbatim** from `Final_LARA_27092026.ipynb`
(sha256 of the class text matches the executed GPU-run copy), so
`Light_Capacity_LARA` in this study is topologically identical to the model
trained in the final LARA study:

* B=32 stem, `T = L` (no bottleneck truncation), six-level **pyramid**
  upsampling decoder,
* a two-layer refinement head instead of a single convolution,
* `latent_dim = 1500 // cr`, so the compression ratio is exact by construction
  and then verified by the pre-flight below.


In [ ]:
class LARA(nn.Module):
    """
    LARA - Light Autoencoder for Reconstruction & Analysis.

    Single production configuration: variant='pyramid_funnel32' (see VARIANT in the CONFIG
    cell). Deep residual + channel-attention encoder, filled projection funnel, learned
    channel-pyramid decoder, conv refinement head.

    variant='pyramid_funnel32' (LARA): FILLS THE FUNNEL - T = L (no min(latent, L) clamp),
                               so the projection width no longer collapses as CR grows, and
                               BOTTLE_CHANNELS=32 -> 6016 projection columns (cr<=30) /
                               3008 (cr>30). cr<10 is identical to 'pyramid'.
    Compression ratio is preserved exactly: latent == input_len // cr (asserted).
    MSE loss only -> fair against the verbatim baseline recipe.

    Other internal tokens kept for provenance / ablations:
    variant='pyramid'   : single latent path; channel pyramid (16->32->64) + refinement head.
    variant='pyramid_v1': exact original 'pyramid' behaviour (archival alias).
    variant='pyramid_b32'    : 32-channel bottleneck spine for cr>=10 (16 for cr<10).
    variant='pyramid_b32_hd' : b32 + high-CR decoder polish for cr>=30 (ceiling 96, k9).
    variant='multiscale': latent split coarse (ceil half) + detail (rest), finest feature
                          carried in its own code, both decoded and fused.
    variant='pyramid_funnel64': same T = L fix with B=64 (cr<=30) / 128 (cr>30) -> 12032
                               columns at EVERY CR (baseline-parity width).
    """

    BOTTLE_CHANNELS = 16

    def __init__(self, cr, input_len=1500, variant="pyramid"):
        super().__init__()
        self.cr = cr
        self.input_len = input_len
        self.variant = variant
        self.latent_dim = input_len // cr

        # ---- variant knobs ----
        b32 = variant in ("pyramid_b32", "pyramid_b32_hd") and cr >= 10
        hd = variant == "pyramid_b32_hd" and cr >= 30
        funnel = variant in ("pyramid_funnel32", "pyramid_funnel64") and cr >= 10
        if variant == "pyramid_funnel32" and funnel:
            self.BOTTLE_CHANNELS = 32
        elif variant == "pyramid_funnel64" and funnel:
            self.BOTTLE_CHANNELS = 64 if cr <= 30 else 128
        else:
            self.BOTTLE_CHANNELS = 32 if b32 else 16
        self.pyramid_ceiling = max(64, self.BOTTLE_CHANNELS) if funnel else (96 if hd else 64)
        self.final_kernel = 9 if hd else 5

        self.L = 188 if cr <= 30 else 94
        self.T = self.L if funnel else min(self.latent_dim, self.L)

        if variant == "multiscale":
            self.latent_coarse = (self.latent_dim + 1) // 2
            self.latent_detail = self.latent_dim - self.latent_coarse
            self.T2 = min(32, max(2, self.latent_dim // 8))
        else:
            self.latent_coarse = self.latent_dim
            self.latent_detail = 0
            self.T2 = None
        assert self.latent_coarse + self.latent_detail == self.latent_dim, "CR must stay exact"

        # ---- encoder stages ----
        self.stage0 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ELU(inplace=True),
            AttentionBlock(32),
        )
        self.stage1 = EnhancedResidualBlock(32, 64, stride=2, use_attention=True)
        self.stage2 = EnhancedResidualBlock(64, 128, stride=2, use_attention=True)
        self.stage3 = EnhancedResidualBlock(128, 256, stride=2, use_attention=True) if cr > 30 else None
        final_channels = 256 if self.stage3 is not None else 128

        # ---- coarse bottleneck (original Light path) ----
        self.bot_conv = nn.Sequential(
            nn.Conv1d(final_channels, self.BOTTLE_CHANNELS, kernel_size=1),
            nn.ELU(inplace=True),
        )
        self.bot_pool = nn.Sequential(
            nn.AdaptiveAvgPool1d(self.T),
            nn.Flatten(),
        )
        self.coarse_head = nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_coarse)

        # ---- detail bottleneck (multiscale only): finest 32ch @ 750 feature ----
        self.detail_block = nn.Sequential()
        self.detail_head = nn.Identity()
        if variant == "multiscale":
            self.detail_block = nn.Sequential(
                nn.Conv1d(32, 32, kernel_size=3, padding=1),
                nn.BatchNorm1d(32),
                nn.ELU(inplace=True),
                nn.AdaptiveAvgPool1d(self.T2),
                nn.Flatten(),
            )
            self.detail_head = nn.Linear(32 * self.T2, self.latent_detail)

        # ---- decoders (channel pyramid + learned upsampling) ----
        self.coarse_decoder = nn.Sequential(
            nn.Linear(self.latent_coarse, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)),
            nn.ELU(inplace=True),
            self._pyramid(self.BOTTLE_CHANNELS, self.T),
        )
        self.detail_decoder = nn.Sequential()
        if variant == "multiscale":
            self.detail_decoder = nn.Sequential(
                nn.Linear(self.latent_detail, self.BOTTLE_CHANNELS * self.T2),
                nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T2)),
                nn.ELU(inplace=True),
                self._pyramid(self.BOTTLE_CHANNELS, self.T2),
            )

        # ---- refinement head ----
        in_ch = 128 if variant == "multiscale" else self.pyramid_ceiling
        self.head = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=5, padding=2),
            nn.ELU(inplace=True),
            nn.Conv1d(32, 1, kernel_size=15, padding=7),
            nn.Sigmoid(),
        )

        self.test_dimensions()

    # ------------------------------------------------------------------ #
    # encoding / decoding
    # ------------------------------------------------------------------ #
    def encoder(self, x):
        """Concatenated latent (coarse + detail) - the actual transmitted code."""
        zc = self.encode_coarse(x)
        if self.variant == "multiscale":
            zd = self.encode_detail(x)
            return torch.cat([zc, zd], dim=-1)
        return zc

    def encode_coarse(self, x):
        h = self.stage0(x)
        h = self.stage1(h)
        h = self.stage2(h)
        if self.stage3 is not None:
            h = self.stage3(h)
        f = self.bot_conv(h)
        f = self.bot_pool(f)
        return self.coarse_head(f)

    def encode_detail(self, x):
        f = self.stage0(x)
        d = self.detail_block(f)
        return self.detail_head(d)

    def forward(self, x):
        zc = self.encode_coarse(x)
        cf = self.coarse_decoder(zc)
        if self.variant == "multiscale":
            zd = self.encode_detail(x)
            df = self.detail_decoder(zd)
            fused = torch.cat([cf, df], dim=1)
            return self.head(fused)
        return self.head(cf)

    # ------------------------------------------------------------------ #
    # helpers
    # ------------------------------------------------------------------ #
    def _pyramid(self, start_channels, start_len):
        """Channel-pyramid upsampler from start_len to input_len using learned
        (nearest + conv) upsampling. Channel ramp up to the variant ceiling."""
        nlev = 6
        ratio = self.input_len / start_len
        levels = []
        cch = start_channels
        prev_len = start_len
        for k in range(nlev):
            tgt = int(round(start_len * (ratio ** ((k + 1) / nlev))))
            tgt = max(tgt, prev_len + 1)
            if k == nlev - 1:
                tgt = self.input_len
            ksize = self.final_kernel if k == nlev - 1 else 5
            nc = min(self.pyramid_ceiling, int(cch * 2))
            levels.append(nn.Sequential(
                nn.Upsample(size=tgt, mode='nearest'),
                nn.Conv1d(cch, nc, kernel_size=ksize, padding=ksize // 2, bias=False),
                nn.BatchNorm1d(nc),
                nn.ELU(inplace=True),
            ))
            cch = nc
            prev_len = tgt
        return nn.Sequential(*levels)

    def test_dimensions(self):
        try:
            with torch.no_grad():
                test_input = torch.randn(1, 1, self.input_len)
                zc = self.encode_coarse(test_input)
                assert zc.numel() == self.latent_coarse, f"coarse latent {zc.numel()} != {self.latent_coarse}"
                if self.variant == "multiscale":
                    zd = self.encode_detail(test_input)
                    assert zd.numel() == self.latent_detail, f"detail latent {zd.numel()} != {self.latent_detail}"
                total = self.encoder(test_input).numel()
                assert total == self.latent_dim, f"total latent {total} != {self.latent_dim}"
                decoded = self(test_input)
                assert decoded.shape[2] == self.input_len, f"output {decoded.shape[2]} != {self.input_len}"
                actual_cr = self.input_len / total
                print(f"[OK] Light-HARA-V2 ({self.variant}) CR={self.cr}: Input {self.input_len} -> Latent {total} "
                      f"(coarse {self.latent_coarse} + detail {self.latent_detail}) -> Output {decoded.shape[2]} "
                      f"(True CR = {actual_cr:.2f})")
        except Exception as e:
            print(f"[FAIL] Light-HARA-V2 ({self.variant}) CR={self.cr}: {e}")
            self.create_fallback_architecture()

    def create_fallback_architecture(self):
        print("Creating fallback architecture...")
        self.stage0 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7), nn.ELU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4), nn.ELU(),
        )
        self.stage1 = nn.Identity()
        self.stage2 = nn.Identity()
        self.stage3 = None
        self.bot_conv = nn.Conv1d(64, self.BOTTLE_CHANNELS, kernel_size=1)
        self.bot_pool = nn.Sequential(nn.AdaptiveAvgPool1d(self.T), nn.Flatten())
        self.coarse_head = nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_dim)
        self.latent_coarse = self.latent_dim
        self.latent_detail = 0
        self.variant = "pyramid"
        self.detail_block = nn.Sequential()
        self.detail_head = nn.Identity()
        self.coarse_decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)), nn.ELU(),
            nn.Upsample(size=self.input_len, mode='linear', align_corners=False),
            nn.Conv1d(self.BOTTLE_CHANNELS, 1, kernel_size=15, padding=7), nn.Sigmoid(),
        )
        self.detail_decoder = nn.Sequential()
        self.head = nn.Identity()
        print("[OK] Fallback architecture created")


# =============================
# The five ablation arms
# =============================
# Light_Capacity_LARA IS the LARA model implemented and trained in
# Final_LARA_27092026 -- class LARA above, used as-is with
# variant="pyramid_funnel32".  The two are not separate arms: the figure and
# table legends read Light_Capacity_LARA everywhere.
#
# Light_Capacity_HARA is still defined above for provenance but is
# deliberately NOT registered: the old light arm used T=min(latent, L), a
# linear-up ResBlock decoder and a single head, which is a different topology
# from LARA.  Comparing both would mix "light-capacity" with "LARA" and
# duplicate the comparison already made by the original study.
LARA_VARIANT = "pyramid_funnel32"


def Light_Capacity_LARA(cr):
    """The LARA arm, pinned to the variant used in Final_LARA_27092026."""
    return LARA(cr, variant=LARA_VARIANT)


ABLATION_MODELS = [
    ("Base_Residual_Attention", Base_Residual_Attention),
    ("No_Attention_HARA",        No_Attention_HARA),
    ("Plain_Conv_HARA",          Plain_Conv_HARA),
    ("Light_Capacity_LARA",      Light_Capacity_LARA),
    ("Heavy_Capacity_HARA",      Heavy_Capacity_HARA),
]


## CR pre-flight

Builds all 50 arm/CR combinations and measures each latent width directly -
never assuming the requested CR - so a dimension bug fails in seconds instead
of after the full run. Raises if any combination is more than
`CR_TOL = 0.001` (0.1%) off, or if the latent is not exactly `1500 // cr`.


In [ ]:
# =============================
# CR pre-flight: latent width must equal the actual compression ratio
# =============================
# Runs BEFORE any training. Every arm is built at every CR and its latent is
# measured, so a dimension bug surfaces in seconds instead of after ~18 h.
# All five arms expose `.encoder` (LARA as a method, the HARA arms as an
# nn.Sequential), so the direct read works; the decoder hook is a fallback.

def measure_latent(model, input_len=1500):
    """Return the measured latent width of a neural model (never assumes the CR)."""
    probe = torch.randn(1, 1, input_len, device=device)
    if hasattr(model, "encoder"):
        with torch.no_grad():
            return int(model.encoder(probe).numel()), "encoder"
    captured = {}
    handle = model.decoder.register_forward_pre_hook(
        lambda m, inp: captured.__setitem__("z", int(inp[0].shape[1])))
    try:
        with torch.no_grad():
            model(probe)
    finally:
        handle.remove()
    return captured["z"], "decoder hook"


print("CR pre-flight")
print("=" * 92)
print(f"{'CR':>4} | {'arm':<22} | {'latent':>7} | {'expected':>8} | "
      f"{'actual CR':>9} | {'rel err':>8} | {'probe':<12} | verdict")
print("-" * 92)

rows, failures = [], []
for _cr in CRS:
    for _name, _ctor in ABLATION_MODELS:
        try:
            _m = _ctor(_cr)
        except KeyError:
            print(f"{_cr:>4} | {_name:<22} | skipped: CR not in this arm's architecture table")
            continue
        try:
            _lat, _how = measure_latent(_m)
        except Exception as e:
            failures.append((_cr, _name, -1, -1, float('nan'), f"measure failed: {e}"))
            continue
        _expected = 1500 // _cr
        _actual = 1500 / _lat
        _rel = abs(_actual - _cr) / _cr
        _ok = (_rel <= CR_TOL) and (_lat == _expected)
        rows.append((_cr, _name, _lat, _expected, _actual, _rel))
        if not _ok:
            failures.append((_cr, _name, _lat, _expected, _actual, f"rel err {100 * _rel:.4f}%"))
        print(f"{_cr:>4} | {_name:<22} | {_lat:>7} | {_expected:>8} | "
              f"{_actual:>9.4f} | {100 * _rel:>7.4f}% | {_how:<12} | "
              f"{'PASS' if _ok else 'FAIL'}")

off_grid = [c for c in CRS if 1500 % c]
print("-" * 92)
if rows:
    print(f"{len(rows)} arm/CR combinations, worst relative error "
          f"{100 * max(r[5] for r in rows):.6f}%, tolerance {100 * CR_TOL}%")
print(f"CRs dividing 1500 evenly: {'all' if not off_grid else off_grid}")

if failures:
    detail = "\n".join(f"  CR={f[0]} {f[1]}: latent {f[2]} (expected {f[3]}) - {f[5]}"
                       for f in failures)
    raise RuntimeError(f"CR pre-flight FAILED ({len(failures)} of {len(rows)}):\n{detail}")
print("CR pre-flight: PASS")


## Training and evaluation

The trainer is the original uniform-budget function with every literal pinned
to the `CONFIG` block: Adam `lr=2e-4`, `weight_decay=1e-5`, gradient clip 1.0,
`ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)`, early stopping 8,
120 epochs max. Early stopping decides the effective duration, so all five arms
get the same budget.

Evaluation reports **per-sample** statistics, matching `Final_LARA_27092026`:
SNR is `10*log10(power_i / mse_i)` per trace and then averaged, alongside
`SNR_median` and the 25th/75th percentiles. There is no Wavelet branch because
none of the five arms is a wavelet model, and `Reconstructions` is stripped
before the CSV is written.


In [ ]:
# =============================
# 🔹  TRAINING CONFIGURATION (UNIFORM BUDGET FROM HARA)
# =============================
def train_model_heavy_ablation(model, train_loader, val_loader, model_name, cr, max_epochs=None):
    """UNIFORM TRAINING BUDGET FOR HEAVY MODEL (120 EPOCHS MAX, EARLY STOPPING DECIDES DURATION)"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training {model_name} CR={cr} on {device} - USING UNIFORM 120-EPOCH BUDGET")

    # UNIFORM EPOCH BUDGET - early stopping decides the effective duration
    if max_epochs is None:
        max_epochs = MAX_EPOCHS

    print(f"Using {max_epochs} epochs max for CR={cr} (uniform budget, early stopping decides)")

    model = model.to(device)

    lr = LR

    print(f"Using learning rate: {lr} (EXACT Ablation Study Value)")

    # EXACT ABLATION STUDY OPTIMIZER CONFIGURATION
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )

    # EXACT ABLATION STUDY SCHEDULER
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=SCHED_PATIENCE,
        min_lr=MIN_LR
    )

    # EXACT ABLATION STUDY EARLY STOPPING CONFIG
    best_val_loss = float('inf')
    patience = PATIENCE
    patience_counter = 0
    best_model_state = None

    train_losses = []
    val_losses = []

    print(f"Starting Heavy Model training with {max_epochs} epoch cap, LR={lr}, patience={patience}")

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            inputs = batch.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)

            loss = nn.MSELoss()(inputs, outputs)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss += loss.item()

            if batch_idx % 500 == 0:
                print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.6f}")

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch.to(device)
                outputs = model(inputs)
                loss = nn.MSELoss()(inputs, outputs)
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        scheduler.step(avg_val_loss)

        print(f"Epoch {epoch+1}/{max_epochs} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

        # EARLY STOPPING CHECK
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), best_file(model_name, cr))
            print(f"✓ New best model saved (val_loss: {best_val_loss:.6f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        if epoch % 2 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"✓ Loaded best model with val_loss: {best_val_loss:.6f}")
    elif os.path.exists(best_file(model_name, cr)):
        model.load_state_dict(torch.load(best_file(model_name, cr), map_location=device))
        print("✓ Loaded saved model from file")

    return model, train_losses, val_losses

In [ ]:
# =============================
# 🔹 Evaluation Metrics (PSNR and SSIM)
# =============================
def calculate_psnr(original, reconstructed):
    """Calculate PSNR between original and reconstructed signals"""
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    psnr = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr

def calculate_ssim_1d(original, reconstructed, data_range=1.0):
    """Calculate SSIM for 1D signals"""
    if len(original.shape) > 1:
        original = original.flatten()
    if len(reconstructed.shape) > 1:
        reconstructed = reconstructed.flatten()

    return ssim(original, reconstructed, data_range=data_range)

# =============================
# 🔹 Model Size Calculation
# =============================
def calculate_model_size(model):
    """Calculate model size in MB"""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

# =============================
# 🔹 Model Evaluation Function
# =============================


# =============================
# Evaluation metrics
# =============================
# Per-sample statistics, matching Final_LARA_27092026 so the two studies are
# directly comparable.  This changes the SNR definition from the original
# ablation: the mean of per-trace 10*log10(power_i / mse_i) rather than
# 10*log10(mean(x^2) / mean((x-x_hat)^2)) over the whole flattened set.
# There is no Wavelet branch here - none of the five arms is a wavelet model.

def evaluate_model(model, test_data, cr, model_name):
    """Evaluate on the held-out test traces and return a metrics dict."""
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(test_data, dtype=torch.float32).to(device)
        recon = model(inputs).cpu().numpy()
        x_processed = inputs.cpu().numpy()[:, 0, :]

    if recon.ndim == 3:
        recon = recon[:, 0, :]

    n_samples = x_processed.shape[0]
    mse_l, snr_l, corr_l, mae_l, psnr_l, ssim_l, grad_l = [], [], [], [], [], [], []

    for i in range(n_samples):
        orig, rec = x_processed[i], recon[i]
        m = np.mean((orig - rec) ** 2)
        signal_power = np.mean(orig ** 2)
        snr_i = 10 * math.log10(signal_power / (m + 1e-12)) if m > 0 else float('inf')
        if np.std(orig) > 0 and np.std(rec) > 0:
            corr_i = np.corrcoef(orig, rec)[0, 1]
        else:
            corr_i = 0.0
        mse_l.append(m)
        snr_l.append(snr_i)
        corr_l.append(corr_i)
        mae_l.append(np.mean(np.abs(orig - rec)))
        psnr_l.append(calculate_psnr(orig, rec))
        ssim_l.append(calculate_ssim_1d(orig, rec))
        grad_l.append(float(np.mean(np.abs(np.diff(orig) - np.diff(rec)))))

    snr_values = np.asarray(snr_l, dtype=float)
    snr_finite = snr_values[np.isfinite(snr_values)]
    ssim_values = np.asarray(ssim_l, dtype=float)

    def _m(arr):
        arr = np.asarray(arr, dtype=float)
        return float(np.mean(arr)), float(np.std(arr))

    mse, mse_std = _m(mse_l)
    corr, corr_std = _m(corr_l)
    mae, mae_std = _m(mae_l)
    psnr, psnr_std = _m(psnr_l)
    grad1, grad1_std = _m(grad_l)

    # ---- achieved compression ratio: measured, never assumed ----
    latent, probe_kind = measure_latent(model)
    actual_cr = 1500 / latent
    cr_error = abs(actual_cr - cr) / cr * 100

    return {
        "MSE": mse, "MSE_std": mse_std,
        "SNR": float(np.mean(snr_finite)) if snr_finite.size else float('inf'),
        "SNR_std": float(np.std(snr_finite)) if snr_finite.size else 0.0,
        "SNR_median": float(np.median(snr_finite)) if snr_finite.size else float('inf'),
        "SNR_P25": float(np.percentile(snr_finite, 25)) if snr_finite.size else float('inf'),
        "SNR_P75": float(np.percentile(snr_finite, 75)) if snr_finite.size else float('inf'),
        "Correlation": corr, "Correlation_std": corr_std,
        "MAE": mae, "MAE_std": mae_std,
        "PSNR": psnr, "PSNR_std": psnr_std,
        "SSIM": float(np.mean(ssim_values)),
        "SSIM_std": float(np.std(ssim_values)),
        "SSIM_median": float(np.median(ssim_values)),
        "SSIM_P25": float(np.percentile(ssim_values, 25)),
        "SSIM_P75": float(np.percentile(ssim_values, 75)),
        "GradL1": grad1, "GradL1_std": grad1_std,
        "CR_Error": cr_error,
        "Model_Size_MB": calculate_model_size(model),
        "Parameter_Count": sum(p.numel() for p in model.parameters()),
        "Model": model_name,
        "CR": cr,
        "Actual_CR": actual_cr,
        "Reconstructions": (x_processed, recon),
    }


## Figures and the main run

The figure code is verbatim from the original study. It takes its labels from
the `Model` column of the results frame, so the legends, per-model folders,
titles and filenames all read `Light_Capacity_LARA` with no plotting edit.

`main()` then reports, for every arm: achieved CR against requested CR, a
summary table, parameter efficiency, and a paired sign test of median SNR
against `Heavy_Capacity_HARA` across the ten matched CRs.

```python
# CPU smoke test first (100 traces, 1 CR, 2 epochs, all five arms)
import os; os.environ["FINAL_ABLATION_SMOKE"] = "1"
```

Expect roughly 18 h on an RTX 4080 SUPER.


In [ ]:
# =============================
# 🔹 Visualization Functions
# =============================
def plot_waveforms_comprehensive(original, reconstructed, cr, model_name, save_figures=True):
    """Enhanced visualization with time and frequency domains for ALL neural network models"""
    num_samples = min(6, len(original))

    fig, axes = plt.subplots(num_samples, 2, figsize=(15, 3*num_samples))

    if num_samples == 1:
        axes = np.array([axes])

    for i in range(num_samples):
        orig_signal = original[i].flatten() if len(original[i].shape) > 1 else original[i]
        recon_signal = reconstructed[i].flatten() if len(reconstructed[i].shape) > 1 else reconstructed[i]

        min_len = min(len(orig_signal), len(recon_signal))
        orig_signal = orig_signal[:min_len]
        recon_signal = recon_signal[:min_len]

        axes[i, 0].plot(orig_signal, 'b-', label='Original', linewidth=1.5, alpha=0.8)
        axes[i, 0].plot(recon_signal, 'r-', label='Reconstructed', linewidth=1.5, alpha=0.8)
        axes[i, 0].set_title(f'{model_name} - CR={cr} - Sample {i+1}', fontsize=12, fontweight='bold')
        axes[i, 0].set_xlabel('Time Samples')
        axes[i, 0].set_ylabel('Amplitude (Normalized)')
        axes[i, 0].legend()
        axes[i, 0].grid(True, alpha=0.3)

        try:
            orig_fft = np.abs(np.fft.fft(orig_signal))
            recon_fft = np.abs(np.fft.fft(recon_signal))
            freq = np.fft.fftfreq(len(orig_signal))

            positive_freq_mask = freq >= 0
            freq_positive = freq[positive_freq_mask]
            orig_fft_positive = orig_fft[positive_freq_mask]
            recon_fft_positive = recon_fft[positive_freq_mask]

            axes[i, 1].semilogy(freq_positive, orig_fft_positive, 'b-', label='Original', linewidth=1.5, alpha=0.8)
            axes[i, 1].semilogy(freq_positive, recon_fft_positive, 'r-', label='Reconstructed', linewidth=1.5, alpha=0.8)
            axes[i, 1].set_title('Frequency Spectrum', fontsize=12, fontweight='bold')
            axes[i, 1].set_xlabel('Frequency (Normalized)')
            axes[i, 1].set_ylabel('Magnitude (log scale)')
            axes[i, 1].legend()
            axes[i, 1].grid(True, alpha=0.3)
        except Exception as e:
            print(f"Warning: Frequency plot failed for sample {i+1}: {e}")
            axes[i, 1].text(0.5, 0.5, f'Frequency plot failed\n{str(e)}',
                           ha='center', va='center', transform=axes[i, 1].transAxes)
            axes[i, 1].set_title('Frequency Spectrum - Failed', fontsize=12, fontweight='bold')

    plt.tight_layout()

    if save_figures:
        model_dir = os.path.join(results_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)

        filename_base = f"{model_name}_CR{cr}_comprehensive"
        png_path = os.path.join(model_dir, f"{filename_base}.png")
        eps_path = os.path.join(model_dir, f"{filename_base}.eps")

        plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.savefig(eps_path, format='eps', bbox_inches='tight', facecolor='white')
        print(f"✓ Saved comprehensive waveforms to: {png_path} and {eps_path}")

    plt.show()
    plt.close()


def plot_frequency_spectrum_all_models(original, reconstructed, cr, model_name, save_figures=True):
    """Save frequency spectrum plots for ALL models (including autoencoders)"""
    num_samples = min(6, len(original))

    for i in range(num_samples):
        try:
            orig_signal = original[i].flatten() if len(original[i].shape) > 1 else original[i]
            recon_signal = reconstructed[i].flatten() if len(reconstructed[i].shape) > 1 else reconstructed[i]

            min_len = min(len(orig_signal), len(recon_signal))
            orig_signal = orig_signal[:min_len]
            recon_signal = recon_signal[:min_len]

            plt.figure(figsize=(10, 6))

            orig_fft = np.abs(np.fft.fft(orig_signal))
            recon_fft = np.abs(np.fft.fft(recon_signal))
            freq = np.fft.fftfreq(len(orig_signal))

            positive_freq_mask = freq >= 0
            freq_positive = freq[positive_freq_mask]
            orig_fft_positive = orig_fft[positive_freq_mask]
            recon_fft_positive = recon_fft[positive_freq_mask]

            plt.semilogy(freq_positive, orig_fft_positive, 'b-', label='Original', linewidth=2, alpha=0.8)
            plt.semilogy(freq_positive, recon_fft_positive, 'r-', label='Reconstructed', linewidth=2, alpha=0.8)
            plt.title(f'{model_name} - CR={cr} - Sample {i+1} - Frequency Spectrum', fontsize=14, fontweight='bold')
            plt.xlabel('Frequency (Normalized)', fontsize=12)
            plt.ylabel('Magnitude (log scale)', fontsize=12)
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)

            if save_figures:
                freq_dir = os.path.join(results_dir, model_name, "frequency_spectrum")
                os.makedirs(freq_dir, exist_ok=True)

                filename_base = f"{model_name}_CR{cr}_sample{i+1}_freq_spectrum"
                png_path = os.path.join(freq_dir, f"{filename_base}.png")
                eps_path = os.path.join(freq_dir, f"{filename_base}.eps")

                plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
                plt.savefig(eps_path, format='eps', bbox_inches='tight', facecolor='white')
                print(f"✓ Saved frequency spectrum for sample {i+1} to: {png_path}")

            plt.close()

        except Exception as e:
            print(f"Error creating frequency spectrum for {model_name} CR={cr} sample {i+1}: {e}")
            plt.close()


def plot_training_history(train_losses, val_losses, model_name, cr, save_figures=True):
    """Plot training and validation loss history"""
    if not train_losses or len(train_losses) == 0:
        print(f"No training history available for {model_name} CR={cr}")
        return

    plt.figure(figsize=(12, 8))
    epochs = range(1, len(train_losses) + 1)

    plt.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2, alpha=0.8)
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2, alpha=0.8)

    plt.title(f'{model_name} - CR={cr} - Training History', fontsize=16, fontweight='bold')
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('MSE Loss', fontsize=14)
    plt.yscale('log')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)

    final_train_loss = train_losses[-1]
    final_val_loss = val_losses[-1] if val_losses else train_losses[-1]
    plt.annotate(f'Final Train: {final_train_loss:.6f}\nFinal Val: {final_val_loss:.6f}',
                xy=(0.02, 0.98), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8),
                fontsize=10, ha='left', va='top')

    if save_figures:
        model_dir = os.path.join(results_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)

        filename_base = f"{model_name}_CR{cr}_training_history"
        png_path = os.path.join(model_dir, f"{filename_base}.png")
        eps_path = os.path.join(model_dir, f"{filename_base}.eps")

        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"✓ Saved training history to: {png_path}")

    plt.show()
    plt.close()


def plot_metrics_comparison(results_df, save_figures=True):
    """Compare all ablation variants across metrics for each CR"""
    models = results_df['Model'].unique()
    metrics = ['SNR', 'Correlation', 'PSNR', 'SSIM', 'MSE', 'MAE']

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        for model in models:
            model_data = results_df[results_df['Model'] == model]
            if metric in ['MSE', 'MAE']:
                axes[i].loglog(model_data['CR'], model_data[metric], 'o-', label=model, markersize=8, linewidth=2)
            else:
                axes[i].semilogx(model_data['CR'], model_data[metric], 'o-', label=model, markersize=8, linewidth=2)

        axes[i].set_title(f'{metric} vs Compression Ratio', fontsize=14, fontweight='bold')
        axes[i].set_xlabel('Compression Ratio', fontsize=12)
        axes[i].set_ylabel(metric, fontsize=12)
        axes[i].grid(True, alpha=0.3)
        axes[i].legend(fontsize=10)

    plt.tight_layout()

    if save_figures:
        png_path = os.path.join(results_dir, "ablation_metrics_comparison.png")
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        print(f"✓ Saved ablation metrics comparison to: {png_path}")

    plt.show()
    plt.close()

In [ ]:
# =============================
# Main ablation study
# =============================
def main():
    global results_df

    save_figures = True

    dataset = EfficientSeismicDataset(waveforms)
    total_samples = len(dataset)

    # 80/10/10, two-stage so the test set is independent (as in the original study)
    test_pool_size = int(0.10 * total_samples)
    val_pool_size = int(0.10 * total_samples)
    train_val_dataset, test_dataset = train_test_split(
        dataset, test_size=test_pool_size, random_state=SEED)
    train_dataset, val_dataset = train_test_split(
        train_val_dataset, test_size=val_pool_size, random_state=SEED)

    n_eval = min(N_EVAL, len(test_dataset))
    test_data = np.array([test_dataset[i] for i in range(n_eval)])

    print(f"\n[split] train={len(train_dataset)}  val={len(val_dataset)}  "
          f"test={n_eval} of {len(test_dataset)} pool  (80/10/10, SEED={SEED})")
    print(f"[split] evaluating on {n_eval} traces (N_EVAL={N_EVAL}); "
          f"test set used only for final evaluation")
    print(f"[split] component {COMPONENT}, magnitude > {MIN_MAGNITUDE}, "
          f"distance_km <= {MAX_DISTANCE_KM}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,
                              num_workers=WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH, shuffle=False,
                              num_workers=WORKERS, pin_memory=True)

    all_results, training_histories, execution_times = [], {}, {}
    study_start = time.time()

    for cr in CRS:
        print(f"\n{'=' * 70}\nCOMPRESSION RATIO: CR={cr}\n{'=' * 70}")

        for model_name, model_class in ABLATION_MODELS:
            print(f"\n--- {model_name} @ CR={cr} ---")
            torch.cuda.empty_cache() if torch.cuda.is_available() else None

            model = model_class(cr).to(device)
            n_params = sum(p.numel() for p in model.parameters())
            latent, _ = measure_latent(model)
            print(f"    params={n_params:,}  latent={latent}  "
                  f"actual CR={1500 / latent:.4f}")

            start = time.time()
            model, train_losses, val_losses = train_model_heavy_ablation(
                model, train_loader, val_loader, model_name, cr, max_epochs=MAX_EPOCHS)
            elapsed = time.time() - start
            execution_times[f"{model_name}_CR{cr}"] = elapsed
            training_histories[f"{model_name}_CR{cr}"] = {
                'train_loss': train_losses, 'val_loss': val_losses}
            print(f"    time: {elapsed / 60:.2f} min")

            plot_training_history(train_losses, val_losses, model_name, cr, save_figures=True)

            metrics = evaluate_model(model, test_data, cr, model_name)
            all_results.append(metrics)
            results_df = pd.DataFrame(all_results)

            # ---- the one line the figure legends key off ----
            print(f"    MSE={metrics['MSE']:.6f}  SNR={metrics['SNR']:.2f}  "
                  f"Corr={metrics['Correlation']:.4f}  "
                  f"CR={metrics['Actual_CR']:.4f} (err {metrics['CR_Error']:.4f}%)")

            if save_figures:
                orig, rec = metrics["Reconstructions"]
                plot_waveforms_comprehensive(orig, rec, cr, model_name, save_figures=True)
                plot_frequency_spectrum_all_models(orig, rec, cr, model_name, save_figures=True)

    total_elapsed = time.time() - study_start
    print(f"\n{'=' * 70}\nTOTAL STUDY TIME: {total_elapsed / 3600:.2f} h\n{'=' * 70}")

    # drop the bulky per-sample arrays before anything touches the frame:
    # they are only needed while figures are being drawn inside the loop
    results_df = results_df.drop(columns=["Reconstructions"], errors="ignore")

    # ---------------- achieved CR: measured, not assumed ----------------
    results_df["CR_Exact"] = results_df["CR_Error"] <= (100 * CR_TOL)
    print("\nAchieved compression ratio (must match the requested CR)")
    print(f"{'CR':>4} | {'arm':<22} | {'actual CR':>9} | {'rel err %':>9} | exact")
    print("-" * 60)
    for _, r in results_df.iterrows():
        rel = abs(r["Actual_CR"] - r["CR"]) / r["CR"] * 100
        print(f"{int(r['CR']):>4} | {r['Model']:<22} | {r['Actual_CR']:>9.4f} | "
              f"{rel:>9.4f} | {r['CR_Exact']}")
    print("-" * 60)
    print(f"exact for {int(results_df['CR_Exact'].sum())}/{len(results_df)} combinations; "
          f"worst relative error {results_df['CR_Error'].max():.6f}%")
    if ENFORCE_CR_EXACT and not results_df["CR_Exact"].all():
        raise RuntimeError("a trained arm did not reproduce its requested CR")

    # ---------------- summary ----------------
    print("\n" + "=" * 70)
    print("ABLATION SUMMARY")
    print("=" * 70)
    summary_df = results_df.groupby('Model').agg(
        Avg_SNR=('SNR', 'mean'),
        Median_SNR=('SNR_median', 'mean'),
        Avg_Correlation=('Correlation', 'mean'),
        Avg_SSIM=('SSIM', 'mean'),
        Median_SSIM=('SSIM_median', 'mean'),
        Avg_MSE=('MSE', 'mean'),
        Avg_Params=('Parameter_Count', 'mean'),
        Avg_Model_MB=('Model_Size_MB', 'mean'),
    ).round(4)
    summary_df["Total_Min"] = [
        round(sum(execution_times.get(f"{m}_CR{c}", 0) for c in CRS) / 60, 2)
        for m in summary_df.index
    ]
    print(summary_df.to_string())
    print("\nBy CR (mean SNR / dB):")
    print(results_df.pivot_table(index='Model', columns='CR', values='SNR').round(2).to_string())

    # ---------------- parameter efficiency ----------------
    print("\n" + "=" * 70)
    print("PARAMETER EFFICIENCY  (mean over the 10 CRs)")
    print("=" * 70)
    eff = (results_df.groupby('Model')['Parameter_Count'].mean() / 1e6).round(3)
    heavy_params = results_df[results_df['Model'] == 'Heavy_Capacity_HARA']['Parameter_Count'].mean()
    eff_rel = (results_df.groupby('Model')['Parameter_Count'].mean() / heavy_params * 100)
    dB100 = results_df[results_df['CR'] == 100].set_index('Model')['SNR'].round(2)
    eff_df = pd.DataFrame({"params_M": eff, "pct_of_heavy": eff_rel.round(1), "dB_at_CR100": dB100})
    print(eff_df.to_string())

    # ---------------- paired significance across matched CRs ----------------
    if not SMOKE:
        base = 'Heavy_Capacity_HARA'
        print("\n" + "=" * 70)
        print(f"PAIRED SIGN TEST vs {base}  (per-CR, median SNR, sign-flip exact)")
        print("=" * 70)
        piv = results_df.pivot_table(index='Model', columns='CR', values='SNR_median')
        if base in piv.index and len(piv.columns) > 1:
            for m in piv.index:
                if m == base:
                    continue
                d = (piv.loc[m] - piv.loc[base]).dropna()
                if len(d) == 0 or (d == 0).all():
                    print(f"  {m:<22} vs {base}: no usable differences")
                    continue
                b = int((d > 0).sum())
                c = int((d < 0).sum())
                p = float(stats.binomtest(min(b, c), b + c, 0.5).pvalue) if (b + c) else 1.0
                verdict = "significant" if p < 0.05 else "not significant"
                print(f"  {m:<22} better on {b}/{len(d)} CRs, worse on {c}  "
                      f"p={p:.4f}  ({verdict})")

    # ---------------- save, without the bulky arrays ----------------
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"\nResults saved to {os.path.abspath(RESULTS_CSV)}")
    print(f"rows: {len(results_df)}  columns: {len(results_df.columns)}  "
          f"(Reconstructions stripped: {'Reconstructions' not in results_df.columns})")

    # ---------------- comparison figures ----------------
    # The only comparison figure the original study produced. Its legend is built
    # from `label=model` off results_df['Model'], so it reads Light_Capacity_LARA
    # with no edit to the plotting code.
    plot_metrics_comparison(results_df, save_figures=True)

    results_df.to_csv(RESULTS_CSV, index=False)
    with open(os.path.join(results_dir, "ablation_training_times.txt"), 'w') as fh:
        for k, v in execution_times.items():
            fh.write(f"{k},{v:.2f}\n")
    print("Done.")
    return results_df


# =============================
# Execute
# =============================
print("\nStarting LARA-aligned HARA ablation study")
print(f"arms: {[n for n, _ in ABLATION_MODELS]}")
print(f"CRs : {CRS}")

results_df = main()
